# V2 timing model training (optional Colab)

A CPU runtime is sufficient. Collection and fresh recovery remain on Windows. This trains the 42-feature ranker and 14-feature prefix detector, selects settings on validation data, and leaves test feature files untouched.

Export on Windows first:
```cmd
python research_pipeline.py export --dataset datasets/my-v2-study --output datasets/my-v2-colab.zip
```


In [ ]:
from google.colab import files
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload exactly one exported v2 bundle.'
archive_path = next(iter(uploaded))


In [ ]:
from pathlib import Path
import os, zipfile, json
workspace = Path('/content/timing_research_v2')
workspace.mkdir(exist_ok=False)
with zipfile.ZipFile(archive_path) as archive:
    for entry in archive.infolist():
        if not (workspace / entry.filename).resolve().is_relative_to(workspace.resolve()):
            raise ValueError('Unsafe archive member')
    archive.extractall(workspace)
os.chdir(workspace)
manifest = json.loads(Path('dataset/manifest.json').read_text())
print('Checker:', manifest['config']['checker'])
print('Whole-password splits:', manifest['split_counts'])
print('Rank features:', len(manifest['feature_names']))
print('Detector features:', len(manifest['detector_names']))


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'research_pipeline.py', 'train',
                '--dataset', 'dataset', '--output', 'model.json',
                '--epochs', '20', '--seed', '7'], check=True)
model = json.loads(Path('model.json').read_text())
print('Selected ranker L2:', model['l2'])
print('Detector threshold:', model['detector_threshold'])
print('Adaptive thresholds:', model['sampling_thresholds'])
print('Test used for selection:', model['test_used_for_selection'])


Download the model, place it in `models/colab-v2.json` on Windows, and evaluate using the unchanged source, DLL, and original dataset:
```cmd
python research_pipeline.py evaluate --dataset datasets/my-v2-study --model models/colab-v2.json --output research_results/colab-v2
```
V1 models/datasets are not compatible with v2. Do not choose model/search settings after viewing test results and call the same cohort an untouched test set. Detector and rank scores are not calibrated probabilities.


In [ ]:
files.download('model.json')
